# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Microsoft Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.


## 設定

在運行此筆記本之前，請確保您已完成以下事項：

1. **擁有一個 Microsoft Foundry 專案** 並已部署聊天模型（例如 `gpt-5-mini`）。
2. **已使用 Azure CLI 登入** — 在終端機執行 `az login`。
3. **設定必要的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Microsoft Foundry 專案端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您所部署模型的名稱。

以下儲存格將安裝您所需的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 建立你的第一個智能助理

一個智能助理需要兩樣東西：

- <strong>指令</strong> 用來告訴它 <em>它是誰</em> 以及 <em>如何表現</em>（系統提示）。
- <strong>工具</strong> — 使用 `@tool` 裝飾的 Python 函數，助理可以調用這些工具來獲取資訊或執行操作。

以下我們定義了一個簡單的工具，返回一個受歡迎的度假目的地列表。當用戶要求旅遊建議時，智能助理會使用這個工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 串流回應

為了提供更具互動性的體驗，您可以<strong>串流</strong>代理的回應。代理會隨著文字片段被生成，同時傳送這些片段，而不是等待整個回覆完成後才顯示。在聊天介面中，若想即時顯示輸出，這尤其有用。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## 總結

在本課程中你學到了如何：

- <strong>建立一個提供者</strong>，透過 `FoundryChatClient` 連接到 Microsoft Foundry Agent Service。
- **使用 `@tool` 裝飾器定義工具**，讓代理能呼叫你的 Python 函數。
- <strong>運行代理</strong> 並輸入用戶訊息，印出其回應。
- <strong>串流回應</strong> 以進行即時輸出。

在下一課我們將更深入探討代理框架，學習如何賦予代理更強大的工具和多步推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
